# Evaluate a challenger with OceanBench

This notebook evaluates a sample of the GLONET system (one forecast) on OceanBench.

To evaluate your own system, replace the content of the next cell with the code that opens your forecasts as `challenger_dataset: xarray.Dataset`.

In [ ]:
%%writefile challenger.py
# Open GLONET forecast sample with xarray
from datetime import datetime
import xarray

challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [
        "https://s3.waw3-1.cloudferro.com/oceanbench-bucket/public/ml-forecast-outputs/glonet/20240103.zarr",
    ],
    engine="zarr",
    preprocess=lambda dataset: dataset.rename({"time": "lead_day_index"}).assign({"lead_day_index": range(10)}),
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=True,
).assign(
    {
        "first_day_datetime": [
            datetime.fromisoformat("2024-01-03"),
        ]
    }
)
challenger_dataset

`evaluate_challenger` runs the OceanBench metrics on the challenger dataset, prints each score table and writes them to `challenger.global.scores.json`.

In [ ]:
import oceanbench

oceanbench.evaluate_challenger("challenger.py")

The scores file holds one row per score. For example, the RMSD of variables compared to GLORYS reanalysis, by depth and lead day:

In [ ]:
import json

import pandas

with open("challenger.global.scores.json", encoding="utf-8") as file:
    scores = json.load(file)

rows = pandas.DataFrame(scores["rows"])
rows[(rows["metric"] == "rmsd_of_variables") & (rows["reference"] == "glorys_reanalysis")].pivot_table(
    index=["depth", "variable"], columns="lead_day", values="value", sort=False
)